# QLoRA Fine-Tuning — SEC Filing Extraction (Colab GPU)

Fine-tunes **Llama 3.1 8B** with QLoRA for extracting structured financial data from SEC filings.

**GPU requirement:** T4 (16 GB, Colab free tier) works. A100 (40 GB, Colab Pro) recommended for larger batches.

| Config | T4 16 GB | A100 40 GB |
|---|---|---|
| Batch size | 4 | 8 |
| Grad accum steps | 8 | 4 |
| Effective batch | 32 | 32 |
| Training time (~500 examples, 3 epochs) | ~25 min | ~8 min |

**What this notebook does:**
1. Clones the repo and installs dependencies
2. Detects GPU and prints VRAM/compute capability
3. Generates synthetic SEC filing training data
4. Runs QLoRA fine-tuning (NF4 4-bit + LoRA r=16)
5. Saves the LoRA adapter to Google Drive

## 1. Environment Setup

In [ ]:
%%capture
!pip install torch transformers peft bitsandbytes accelerate datasets trl
!pip install sentencepiece protobuf loguru pyyaml python-dotenv tqdm rich click

import os
REPO_URL = "https://github.com/akuo6/financial-llm.git"
REPO_DIR = "/content/FinDocAnalyzer"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

## 2. GPU Detection

Verifies a CUDA GPU is available and prints specs. If you see "No GPU," go to **Runtime → Change runtime type → GPU**.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected. Go to Runtime → Change runtime type → GPU (T4 or A100)."
    )

gpu = torch.cuda.get_device_properties(0)
vram_gb = gpu.total_mem / 1e9

print(f"GPU:             {gpu.name}")
print(f"VRAM:            {vram_gb:.1f} GB")
print(f"Compute cap:     {gpu.major}.{gpu.minor}")
print(f"CUDA version:    {torch.version.cuda}")
print(f"PyTorch version: {torch.__version__}")

# Auto-configure batch size based on available VRAM
if vram_gb >= 35:
    BATCH_SIZE = 8
    GRAD_ACCUM = 4
    GPU_TIER = "A100"
elif vram_gb >= 20:
    BATCH_SIZE = 6
    GRAD_ACCUM = 6
    GPU_TIER = "L4/A10"
else:
    BATCH_SIZE = 4
    GRAD_ACCUM = 8
    GPU_TIER = "T4"

print(f"\nAuto-config ({GPU_TIER}):")
print(f"  Batch size:      {BATCH_SIZE}")
print(f"  Grad accum:      {GRAD_ACCUM}")
print(f"  Effective batch: {BATCH_SIZE * GRAD_ACCUM}")

## 3. HuggingFace Authentication

You need access to [meta-llama/Llama-3.1-8B](https://huggingface.co/meta-llama/Llama-3.1-8B) (request via the HuggingFace model page).

In [ ]:
from google.colab import userdata
from huggingface_hub import login

try:
    hf_token = userdata.get("HF_TOKEN")
except userdata.SecretNotFoundError:
    hf_token = input("Enter your HuggingFace token: ")

login(token=hf_token)
os.environ["HF_TOKEN"] = hf_token
print("Authenticated with HuggingFace Hub.")

## 4. Generate Training Data

Creates synthetic SEC filing training pairs. In production (Iteration 2), this will be replaced by real EDGAR filings.

In [ ]:
import sys
sys.path.insert(0, REPO_DIR)

!python scripts/download_dataset.py
!python scripts/format_data.py

import json
from pathlib import Path

data_dir = Path(REPO_DIR) / "data"
train_file = data_dir / "sec_filings_train.chat.jsonl"
if train_file.exists():
    n_examples = sum(1 for _ in open(train_file))
    with open(train_file) as f:
        sample = json.loads(f.readline())
    print(f"\nTraining examples: {n_examples}")
    print(f"Sample keys:       {list(sample.keys())}")
else:
    train_file = data_dir / "sec_filings_train.jsonl"
    n_examples = sum(1 for _ in open(train_file))
    print(f"\nTraining examples: {n_examples} (raw format, will format on-the-fly)")

## 5. QLoRA Fine-Tuning

Loads Llama 3.1 8B in NF4 4-bit, injects LoRA adapters (r=16, α=32), and trains with SFTTrainer.

**Memory budget:**
- Base model (NF4): ~7.2 GB
- LoRA adapters (FP16): ~0.2 GB
- Optimizer states (paged AdamW 8-bit): ~0.4 GB
- Activations (gradient checkpointing): ~2–4 GB
- **Total: ~10–12 GB** (fits on T4 16 GB)

In [ ]:
from src.config import load_config
from training.train import (
    create_bnb_config,
    create_lora_config,
    load_base_model,
    prepare_dataset,
    create_training_args,
)
from training.callbacks import MetricsCallback, EarlyStoppingOnLoss

config = load_config()

# Apply Colab-optimized batch settings
config["training"]["batch_size"] = BATCH_SIZE
config["training"]["gradient_accumulation_steps"] = GRAD_ACCUM

print("Training config:")
for k, v in config["training"].items():
    print(f"  {k}: {v}")
print(f"\nLoRA config:")
for k, v in config["lora"].items():
    print(f"  {k}: {v}")

In [ ]:
from peft import get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

# Step 1: Create quantization config
bnb_config = create_bnb_config(config)

# Step 2: Load base model in NF4 4-bit
model, tokenizer = load_base_model(
    config["model"]["base_model"],
    bnb_config,
    config["model"]["max_seq_length"],
)

# Step 3: Inject LoRA adapters
lora_config = create_lora_config(config)
model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"\nTrainable: {trainable / 1e6:.1f}M / {total / 1e6:.0f}M ({100 * trainable / total:.2f}%)")

# GPU memory after model load
mem_gb = torch.cuda.memory_allocated() / 1e9
print(f"GPU memory used: {mem_gb:.2f} GB / {vram_gb:.1f} GB ({100 * mem_gb / vram_gb:.0f}%)")

In [ ]:
# Step 4: Load dataset
dataset = prepare_dataset(
    config["data"]["train_path"],
    tokenizer,
    config["model"]["max_seq_length"],
    config["data"].get("max_train_samples"),
)

# Step 5: Training arguments
OUTPUT_DIR = os.path.join(REPO_DIR, config["training"]["output_dir"])
training_args = create_training_args(config, OUTPUT_DIR)

# Step 6: Trainer
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
    max_seq_length=config["model"]["max_seq_length"],
    callbacks=[
        MetricsCallback(),
        EarlyStoppingOnLoss(patience=5, min_delta=0.01),
    ],
)

# Step 7: Train
print(f"\nStarting training: {config['training']['num_epochs']} epochs, "
      f"{len(dataset)} examples, batch {BATCH_SIZE}×{GRAD_ACCUM}={BATCH_SIZE * GRAD_ACCUM}")
print("=" * 60)

train_result = trainer.train()

# Step 8: Save adapter
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("=" * 60)
print(f"Training complete. Adapter saved to: {OUTPUT_DIR}")
print(f"Final loss: {train_result.metrics.get('train_loss', 'N/A')}")

peak_mem_gb = torch.cuda.max_memory_allocated() / 1e9
print(f"Peak GPU memory: {peak_mem_gb:.2f} GB")

## 6. Save Adapter to Google Drive

Colab VMs are ephemeral. Save the trained adapter (~500 MB) to Drive so it persists across sessions.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_SAVE_DIR = "/content/drive/MyDrive/FinDocAnalyzer/models/llama-sec-v1"
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)

import shutil
adapter_files = list(Path(OUTPUT_DIR).glob("*"))
for f in adapter_files:
    dest = os.path.join(DRIVE_SAVE_DIR, f.name)
    if f.is_file():
        shutil.copy2(str(f), dest)

total_mb = sum(f.stat().st_size for f in Path(DRIVE_SAVE_DIR).rglob("*") if f.is_file()) / 1e6
print(f"Adapter saved to Google Drive: {DRIVE_SAVE_DIR}")
print(f"Total size: {total_mb:.1f} MB")
print(f"Files: {[f.name for f in Path(DRIVE_SAVE_DIR).iterdir() if f.is_file()]}")

## 7. Training Metrics Visualization

In [ ]:
import matplotlib.pyplot as plt

metrics_path = os.path.join(OUTPUT_DIR, "training_log.json")

if os.path.exists(metrics_path):
    with open(metrics_path) as f:
        log = json.load(f)

    steps = [e["step"] for e in log if "loss" in e]
    losses = [e["loss"] for e in log if "loss" in e]
    lrs = [e.get("learning_rate", 0) for e in log if "loss" in e]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(steps, losses, linewidth=1.5, color="#2563eb")
    ax1.set_xlabel("Step")
    ax1.set_ylabel("Loss")
    ax1.set_title("Training Loss")
    ax1.grid(True, alpha=0.3)

    ax2.plot(steps, lrs, linewidth=1.5, color="#dc2626")
    ax2.set_xlabel("Step")
    ax2.set_ylabel("Learning Rate")
    ax2.set_title("Learning Rate Schedule (Cosine)")
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
else:
    print("No training log found (MetricsCallback may not have fired).")

## Next Steps

- Open `inference_eval.ipynb` to run extraction on sample filings and evaluate accuracy
- The adapter on Google Drive can be loaded in future sessions without retraining
- To fine-tune on real SEC filings, replace the synthetic data generation step with EDGAR data (see Iteration 2 roadmap in `CLAUDE.md`)